#exploring workbokk

In [1]:
from openpyxl import load_workbook

wb = load_workbook('Sample-sales-data-excel.xlsx')



In [2]:
print("sheet names in this workbook: ")
print(wb.sheetnames)
print("active sheet: ")
print(wb.active)

import pandas as pd

df =  pd.read_excel("Sample-sales-data-excel.xlsx")
print("\nfile shape is: ",df.shape)
print('\n first 5 rows: \n', df.head())

sheet names in this workbook: 
['Orders']
active sheet: 
<Worksheet "Orders">

file shape is:  (9994, 21)

 first 5 rows: 
    Row ID        Order ID Order Date  Ship Date       Ship Mode Customer ID  \
0       1  CA-2016-152156 2016-11-08 2016-11-11    Second Class    CG-12520   
1       2  CA-2016-152156 2016-11-08 2016-11-11    Second Class    CG-12520   
2       3  CA-2016-138688 2016-06-12 2016-06-16    Second Class    DV-13045   
3       4  US-2015-108966 2015-10-11 2015-10-18  Standard Class    SO-20335   
4       5  US-2015-108966 2015-10-11 2015-10-18  Standard Class    SO-20335   

     Customer Name    Segment        Country             City  ...  \
0      Claire Gute   Consumer  United States        Henderson  ...   
1      Claire Gute   Consumer  United States        Henderson  ...   
2  Darrin Van Huff  Corporate  United States      Los Angeles  ...   
3   Sean O'Donnell   Consumer  United States  Fort Lauderdale  ...   
4   Sean O'Donnell   Consumer  United States  Fort 

In [3]:
print("=========== column names===========")
print(df.columns)

=========== column names===========
Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State',
       'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category',
       'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit'],
      dtype='object')


In [ ]:
print('duplicates count      :',df.duplicated().sum())
print('\nmissing values count: ',df.isnull().sum())
print('\n ',df.info())


duplicates count      : 0

missing values count:  Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Row ID         9994 non-null   int64         
 1   Order ID       9994 non-null   object        
 2   Order Date     9994 non-null   datetime64[ns]
 3   Ship Date      9994 non-null   datetime64[ns]
 4   Ship Mode      9994 non-null   object        
 5   Customer ID    9994 non-null   object        
 6   Customer Name  999

In [ ]:
print('count of negative Quantity: ',(df['Quantity']<0).sum())
print('count of negative Sales: ',(df['Sales']<0).sum())
print('count of negative profit: ',(df['Profit']<0).sum())

count of negative Quantity:  0
count of negative Sales:  0
count of negative profit:  1871


In [ ]:
print('\n number of categories: ',df['Category'].nunique())
print('\n number of regions   : ', df['Region'].nunique())



 number of categories:  3

 number of regions   :  4


In [5]:
from IPython.display import display

#==============================================
# CATEGORY SUMMARY
#==============================================

category_summary = (df.groupby('Category')
                    .agg(total_sales=('Sales','sum'),
                         total_profit=('Profit','sum'))
                    .sort_values('total_sales', ascending=False))

print('\n================ category summary ================\n')
display(category_summary.style.format("{:,.2f}"))

#==============================================
# Region SUMMARY
#==============================================

region_summary = (df.groupby('Region')
                    .agg(total_sales=('Sales','sum'),
                         total_profit=('Profit','sum'))
                    .sort_values('total_sales', ascending=False))

print('\n================ region summary ================\n')
display(region_summary.style.format("{:,.2f}"))

#==============================================
# REPORT SUMMARY
#==============================================

total_sales = df['Sales'].sum()
total_profit = df['Profit'].sum()
avg_discount = df['Discount'].mean()

num_categories = df['Category'].nunique()
total_orders = len(df)
negative_profit = (df['Profit']<0).sum()


================ category summary ================



,total_sales,total_profit
Category,,
Technology,"836,154.03","145,454.95"
Furniture,"741,999.80","18,451.27"
Office Supplies,"719,047.03","122,490.80"



================ region summary ================



,total_sales,total_profit
Region,,
West,"725,457.82","108,418.45"
East,"678,781.24","91,522.78"
Central,"501,239.89","39,706.36"
South,"391,721.91","46,749.43"


# Save the DFs as Excel sheets

In [6]:
with pd.ExcelWriter('Sales_Report.xlsx') as writer:

  category_summary.to_excel(writer, sheet_name='Category Summary')

  region_summary.to_excel(writer, sheet_name='Region Summary')

#Make the report professional

In [7]:
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl import load_workbook
#==============================================
# loading workbook
#==============================================
wb = load_workbook('Sales_Report.xlsx')

print(wb.sheetnames)

category_ws = wb['Category Summary']
region_ws = wb['Region Summary']

#==============================================
# setting values
#==============================================

header_font = Font(bold=True)
header_fill = PatternFill( fill_type='solid', start_color='FFD966')
header_alignment = Alignment(horizontal='center')

#==============================================
# applying the values
#==============================================

for ws in [category_ws,region_ws]:
  for cell in ws[1]:
    cell.font = header_font
    cell.fill = header_fill
    cell.alignment = header_alignment

#==============================================
# numerical values formatting
#==============================================

for ws in [category_ws,region_ws]:
  for row in ws.iter_rows(min_row=2):
    row[1].number_format = "#,##0.00"
    row[2].number_format = '#,##0.00'

#==============================================
# adjusting column size
#==============================================

for ws in [category_ws,region_ws]:
  for column in ws.columns:
    max_length = 0

    for cell in column:
      if cell.value is not None:
        max_length = max(max_length,len(str(cell.value)))

    ws.column_dimensions[column[0].column_letter].width = max_length+2

#==============================================
# freeze the header
#==============================================

for ws in [category_ws,region_ws]:
  ws.freeze_panes = 'A2'

#==============================================
# save
#==============================================

wb.save('Sales_Report.xlsx')

['Category Summary', 'Region Summary']


#creating report summary sheet

In [13]:
if 'Report Summary' in wb.sheetnames:
    del wb['Report Summary']

summary_ws = wb.create_sheet('Report Summary')

summary_ws['A1'] = 'SALES REPORT SUMMARY'
summary_ws['A3'] = 'Total Sales'
summary_ws['B3'] = df['Sales'].sum()

summary_ws['A4'] = 'Total Profit'
summary_ws['B4'] = df['Profit'].sum()

summary_ws['A5'] = 'Average Discount'
summary_ws['B5'] = df['Discount'].mean()

summary_ws['A6'] = 'Total Orders'
summary_ws['B6'] = len(df)

summary_ws['A7'] = 'Number of Categories'
summary_ws['B7'] = df['Category'].nunique()

summary_ws['A8'] = 'Negative Profit Transactions'
summary_ws['B8'] = (df['Profit']<0).sum()

#==============================================
# Formatting Report Summary sheet
#==============================================

summary_ws.merge_cells('A1:B1')
summary_ws['A1'].font = Font(bold=True , size=16)
summary_ws['A1'].alignment = Alignment(horizontal='center')

for row in range(3,9):
  summary_ws[f'A{row}'].font = Font(bold=True)

summary_ws['B3'].number_format = '#,##0.00'
summary_ws['B4'].number_format = '#,##0.00'

summary_ws['B5'].number_format = '0.00%'

#==============================================
# columns width
#==============================================

summary_ws.column_dimensions['A'].width = 30
summary_ws.column_dimensions['B'].width = 18

wb.save('Sales_Report.xlsx')
